# Projektin Nimi (TBD)

Tony Karlin, Onni Kivinen, Joni Heikkilä, Jarkko Kärki

In [85]:
import geopandas as gpd
import pandas as pd

## 1.0 Business understanding

## 2.0 Data understanding

Helsingissä tapahtuneiden liikenneottemuuksien tapahtumapaikat, vakavuustasteet ja onnettomuuslajit vuodesta 2000 alkaen.

In [86]:
geodata_filepath = "datasets/piirialuejako-1995-2019.gpkg"
areas = gpd.read_file(geodata_filepath, layer="osa_alue_2019")

accidents_filepath = "datasets/liikenneonnettomuudet.csv"
project_df = pd.read_csv(accidents_filepath, sep=";", decimal=",")

print("Datasetin rivien määrä:", len(project_df))
project_df.head()

Datasetin rivien määrä: 53800


,LAJI,pohj_etrs,ita_etrs,VAKAV_A,VV
0,JK,6675786.73,25501661.91,1,2022
1,JK,6674533.32,25502790.94,2,2022
2,JK,6679536.47,25506837.81,2,2022
3,JK,6675297.68,25498807.13,2,2022
4,JK,6674924.21,25495525.13,2,2022


### Muuttujat
| Muuttuja  | Selitys             | Lisätiedot|
| --------- | ------------------- |---------|
| LAJI      | onnettomuustyyppi   |jk = jalankulkijaonnettomuus, pp = polkupyöräonnettomuus, mp = mopo/moottoripyöräonnettomuus, ma = moottoriajoneuvo-onnettomuus|
| pohj_etrs | pohjoiskoordinaatti |Pohjoiskoordinaatti ETRS-GK25-järjestelmässä|
| ita_etrs  | itäkoordinaatti     |Itäkoordinaatti ETRS-GK25-järjestelmässä|
| VAKAV_A   | vakavuus            |1 = omaisuusvahinko,  2 = loukkaantumiseen johtanut, 3 = kuolemaan johtanut|
| VV        | vuosi               | Vuosi jolloin onnettomuus tapahtunut|



Tarkistetaan puuttuvien arvojen määrä:

In [87]:
project_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53800 entries, 0 to 53799
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   LAJI       53800 non-null  object
 1   pohj_etrs  53797 non-null  object
 2   ita_etrs   53797 non-null  object
 3   VAKAV_A    53800 non-null  int64 
 4   VV         53800 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 2.1+ MB


In [88]:
project_df.isnull().sum()

LAJI         0
pohj_etrs    3
ita_etrs     3
VAKAV_A      0
VV           0
dtype: int64

## 3.0 Data preparation

Pudotetaan kolme puutteellista riviä.

In [89]:
project_df = project_df.dropna()
project_df.isnull().sum()

LAJI         0
pohj_etrs    0
ita_etrs     0
VAKAV_A      0
VV           0
dtype: int64

In [90]:
accidents = gpd.GeoDataFrame(
    project_df,
    geometry=gpd.points_from_xy(project_df["ita_etrs"], project_df["pohj_etrs"]),
    crs="EPSG:3879"
)
areas = areas.to_crs(accidents.crs)

accidents = gpd.sjoin(
    accidents,
    areas[["Nimi", "geometry"]],
    how="left",
    predicate="within"
)

accidents = accidents.rename(columns={"Nimi": "Osa-alue"})
accidents = accidents.drop(columns=["geometry", "index_right"])

In [91]:
# Sarakkeiden uudellennimeäminen
col_rename = {"LAJI": "O_Tyyppi", "pohj_etrs": "Pohj_coords", "ita_etrs": "Itä_coords","VAKAV_A": "Vakavuus", "VV": "Vuosi"}
accidents = accidents.rename(columns=col_rename)

# Sarakkeiden uudelleenjärjestys
accidents = accidents.iloc[:, [1, 2, 5, 0, 4, 3]]
accidents

,Pohj_coords,Itä_coords,Osa-alue,O_Tyyppi,Vuosi,Vakavuus
0,6675786.73,25501661.91,Länsi-Hertto,JK,2022,1
1,6674533.32,25502790.94,Yliskylä,JK,2022,2
2,6679536.47,25506837.81,Mellunmäki,JK,2022,2
3,6675297.68,25498807.13,Kalasatama,JK,2022,2
4,6674924.21,25495525.13,Taka-Töölö,JK,2022,2
...,...,...,...,...,...,...
53795,6674696.29,25497044.46,Linjat,PP,2024,2
53796,6675329.91,25497183.32,Vallila,PP,2024,1
53797,6675746.02,25494119.16,Pikku Huopal,PP,2024,1
53798,6678387.52,25494869.15,Kivihaka,PP,2024,1


In [92]:
print(accidents.dtypes)

Pohj_coords    object
Itä_coords     object
Osa-alue       object
O_Tyyppi       object
Vuosi           int64
Vakavuus        int64
dtype: object


In [93]:
# Sarakkeiden muuttaminen tyypistä str -> Categorical
cols_to_categorical = ["Osa-alue", "O_Tyyppi"]
accidents[cols_to_categorical] = accidents[cols_to_categorical].astype("category")

cols_to_float = ["Pohj_coords", "Itä_coords"]
accidents[cols_to_float] = accidents[cols_to_float].astype(float)

In [94]:
print(accidents.dtypes)

Pohj_coords     float64
Itä_coords      float64
Osa-alue       category
O_Tyyppi       category
Vuosi             int64
Vakavuus          int64
dtype: object


In [95]:
accidents.describe(include='all')

,Pohj_coords,Itä_coords,Osa-alue,O_Tyyppi,Vuosi,Vakavuus
count,5.379700e+04,5.379700e+04,53786,53797,53797.000000,53797.000000
unique,NaN,NaN,140,4,NaN,NaN
top,NaN,NaN,Kamppi,MA,NaN,NaN
freq,NaN,NaN,3230,44835,NaN,NaN
mean,6.676639e+06,2.549795e+07,NaN,NaN,2009.892931,1.212949
std,3.502269e+03,3.695722e+03,NaN,NaN,6.408570,0.418822
min,6.670495e+06,2.549076e+07,NaN,NaN,2000.000000,1.000000
25%,6.673407e+06,2.549564e+07,NaN,NaN,2004.000000,1.000000
50%,6.676020e+06,2.549701e+07,NaN,NaN,2010.000000,1.000000
75%,6.679532e+06,2.550009e+07,NaN,NaN,2015.000000,1.000000


## 4.0 Modeling

## 5.0 Evaluation

## 6.0 Deployment